# A: intact LMLM detector — Kaggle GPU probe

Use a **private** notebook, Internet on, T4 GPU. Checks the released EXE detector 1923 initialized from MLM 799, with original backbone/head, FP32 and native Mamba kernels. Synthetic bytes/tokens only; no PE samples, no RanDS fitting, no training. The 65,536-token case measures capacity, not a valid RanDS sample or model quality.

Bundle: enumerated source, lock/config and six detector artifacts (including trainer metadata), with per-file size/SHA-256 manifest. Keep staging under `/kaggle/temp`; only logs/reports go to `/kaggle/working`. A failed probe must be investigated before full evaluation. See the maintained A guide for test-only/resume commands and artifact limitations.

In [ ]:
from pathlib import Path
import os

# Supply a private GET capability via environment/Kaggle Secrets; never print it.
BUNDLE_URL = os.environ.get("MALWEAVE_A_BUNDLE_URL", "")
BUNDLE_SHA256 = os.environ.get("MALWEAVE_A_BUNDLE_SHA256", "")
BUNDLE_BYTES = int(os.environ.get("MALWEAVE_A_BUNDLE_BYTES", "0"))
STAGING = Path("/kaggle/temp/malweave-a-probe")
OUTPUT = Path("/kaggle/working/a-probe")
DEVICE = "cuda:0"
if not BUNDLE_URL:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    BUNDLE_URL = secrets.get_secret("MALWEAVE_A_BUNDLE_URL")
    BUNDLE_SHA256 = secrets.get_secret("MALWEAVE_A_BUNDLE_SHA256")
    BUNDLE_BYTES = int(secrets.get_secret("MALWEAVE_A_BUNDLE_BYTES"))


In [ ]:
import hashlib
import json
import shutil
import subprocess
import tarfile
import urllib.request
from pathlib import PurePosixPath
from urllib.parse import urlsplit

os.umask(0o077)
assert not STAGING.exists(), "Use a fresh session/staging directory; do not overwrite inputs."
assert not OUTPUT.exists(), "Inspect the previous probe before starting another."
assert len(BUNDLE_SHA256) == 64 and 0 < BUNDLE_BYTES <= 512 * 1024**2
STAGING.mkdir(parents=True)
OUTPUT.mkdir(parents=True)

def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024**2), b""):
            h.update(block)
    return h.hexdigest()

def download(url, target, *, limit, expected_sha, expected_bytes=None):
    location = urlsplit(url)
    assert location.scheme == "https" and location.netloc and not location.username
    try:
        with urllib.request.urlopen(url, timeout=120) as incoming, target.open("wb") as outgoing:
            size = 0
            while block := incoming.read(1024**2):
                size += len(block)
                assert size <= limit, "Download exceeds its limit."
                outgoing.write(block)
    except Exception:
        raise RuntimeError("Private/worker download failed; check capability expiry and network.") from None
    assert digest(target) == expected_sha, "Download SHA-256 mismatch."
    assert expected_bytes is None or size == expected_bytes, "Download size mismatch."

archive = STAGING / "bundle.tar.gz"
download(BUNDLE_URL, archive, limit=BUNDLE_BYTES, expected_sha=BUNDLE_SHA256, expected_bytes=BUNDLE_BYTES)
# Validate every member before admitting any source code. No extractall, links or traversal.
with tarfile.open(archive, "r:gz") as bundle:
    members = bundle.getmembers()
    assert len({m.name for m in members}) == len(members)
    assert all(m.isfile() and not PurePosixPath(m.name).is_absolute()
               and ".." not in PurePosixPath(m.name).parts for m in members)
    assert sum(m.size for m in members) <= 512 * 1024**2
    manifest = json.load(bundle.extractfile("snapshot-manifest.json"))
    assert manifest["schema"] == 1
    assert {m.name for m in members} == {*manifest["files"], "snapshot-manifest.json"}
    for name, record in manifest["files"].items():
        destination = STAGING / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        with bundle.extractfile(name) as incoming, destination.open("wb") as outgoing:
            shutil.copyfileobj(incoming, outgoing)
        assert destination.stat().st_size == record["bytes"] and digest(destination) == record["sha256"]
archive.unlink()
BUNDLE_URL = ""  # Capability is not part of the exported report.
REPO = STAGING / "repo"
CHECKPOINT = STAGING / "detector"
assert (REPO / "uv.lock").is_file() and (CHECKPOINT / "model.safetensors").is_file()

# Pinned standalone uv; no installation into Kaggle's system Python.
uv_archive = STAGING / "uv.tar.gz"
download("https://github.com/astral-sh/uv/releases/download/0.11.9/uv-x86_64-unknown-linux-gnu.tar.gz",
         uv_archive, limit=64 * 1024**2,
         expected_sha="5c43f82077ff0cd5aec588286cbabd89913e4d045bd4e8aa60b20b3ecffc36e3")
UV = STAGING / "uv"
with tarfile.open(uv_archive, "r:gz") as binary:
    member = binary.getmember("uv-x86_64-unknown-linux-gnu/uv")
    assert member.isfile()
    with binary.extractfile(member) as incoming, UV.open("wb") as outgoing:
        shutil.copyfileobj(incoming, outgoing)
UV.chmod(0o700)
uv_archive.unlink()
ENV = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(STAGING / "env"),
           UV_CACHE_DIR=str(STAGING / "uv-cache"), PYTHONPATH=str(REPO),
           TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_TELEMETRY="1")
# Remove capability variables from child environments too.
for key in ("MALWEAVE_A_BUNDLE_URL", "MALWEAVE_A_BUNDLE_SHA256", "MALWEAVE_A_BUNDLE_BYTES"):
    ENV.pop(key, None)
subprocess.run([str(UV), "python", "install", "3.12.12"], env=ENV, check=True)
subprocess.run([str(UV), "sync", "--locked", "--python", "3.12.12", "--no-default-groups", "--group", "gpu"],
               cwd=REPO, env=ENV, check=True)
print("Verified source/checkpoint bundle; locked GPU environment ready.")


In [ ]:
import time

started = time.time()
command = [str(STAGING / "env/bin/python"), "-m", "malweave", "experiment", "probe-a",
           "--checkpoint", str(CHECKPOINT), "--output-dir", str(OUTPUT / "results"), "--device", DEVICE]
with (OUTPUT / "probe.log").open("w") as log:
    result = subprocess.run(command, cwd=REPO, env=ENV, stdout=log, stderr=subprocess.STDOUT)
receipt = {"bundle_sha256": BUNDLE_SHA256, "bundle_bytes": BUNDLE_BYTES,
           "started_unix": started, "elapsed_seconds": time.time() - started,
           "returncode": result.returncode, "synthetic_only": True, "training": False}
(OUTPUT / "receipt.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))
if result.returncode:
    print((OUTPUT / "probe.log").read_text()[-12000:])
assert result.returncode == 0, "A capacity probe failed; inspect probe.log before full evaluation."


In [ ]:
from IPython.display import Markdown, display

report = json.loads((OUTPUT / "results/probe.json").read_text())
rows = ["| Case | Tokens | Seconds | Peak allocated GiB | Peak reserved GiB |",
        "| --- | ---: | ---: | ---: | ---: |"]
for case in report["cases"]:
    hardware = case["hardware"]
    rows.append(f"| {case['name']} | {case['tokens']:,} | {case['seconds']:.3f} | "
                f"{hardware['peak_allocated_bytes']/2**30:.3f} | {hardware['peak_reserved_bytes']/2**30:.3f} |")
display(Markdown("\n".join(rows)))
print(report["hardware"]["gpu"], report["hardware"]["torch"], report["hardware"]["precision"])
print("Synthetic capacity only: no RanDS quality scores. Results are in", OUTPUT)
